# 0. PCA of Beehive Acoustic Features (per-session & combined)

This notebook:
- loads your feature CSVs (RMS, ZCR, Spectral Bandwidth (SB), Spectral Centroid (SC), Spectral Rolloff)
- loads environmental logs from Excel (temperature & humidity)
- merges on timestamp
- performs PCA:
  - **per session** (loop)
  - **across all sessions combined**
- plots:
  - Scree (variance explained)
  - PC1 vs PC2 colored by **session**
  - PC1 vs PC2 colored by **temperature/humidity**
  - PC time series (PC1/PC2 vs time)
  - Loadings (feature contributions to PCs)

Notes:
- MFCC & Chroma are intentionally excluded (too many dimensions).
- PCA can be purely acoustic, or include temp/hum (toggle below).


# 1. Imports and global settings

In [1]:
import os, glob, re, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

warnings.simplefilter("ignore", category=UserWarning)
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

# 2. Main configuration

In [2]:
# Default timezone for NAIVE timestamps 
DEFAULT_LOCAL_TZ = "Europe/Warsaw"

# Where to save aligned CSVs + figures
OUTPUT_DIR = r"X:\Dissertation Files\Extracted Features\_cleaned_for_PCA_vv"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# Merge tolerances
FEAT_TOL = pd.Timedelta("2s")        # features are ~per-second
ENV_TOL  = pd.Timedelta("120s")      # env logs slower, larger window

# Alignment strategy
ALIGN_MODE = "asof"                  # or "resample"
RESAMPLE_RULE   = "1S"
ENV_FFILL_LIMIT = 120                # seconds (only environment)

# PCA options
ACOUSTIC_COLS      = ["rms","zcr","sb","sc","rolloff"]
INCLUDE_ENV_IN_PCA = False           # True -> include temp, humidity
N_COMPONENTS       = None            # None -> up to 6
SCALE_BEFORE_PCA   = True
SAVE_PCA_READY     = True            # also save "no-NaN" subset used for PCA

# Plot axes mode for PC1/PC2
AXIS_MODE = "global"                 # "global" or "fixed"
FIXED_PC1 = (-1, 1)                  # used only if AXIS_MODE == "fixed"
FIXED_PC2 = (-1, 1)


# 3. Session paths 

In [3]:
SESSIONS = {
    "Set I (13.08.2024-15.08.2024)": {
        "rms"    : r"X:\Dissertation Files\Extracted Features\Set I 13.08.2024 - 15.08.2024\RMS_features",
        "zcr"    : r"X:\Dissertation Files\Extracted Features\Set I 13.08.2024 - 15.08.2024\ZCR_features",
        "sb"     : r"X:\Dissertation Files\Extracted Features\Set I 13.08.2024 - 15.08.2024\SB_features",
        "sc"     : r"X:\Dissertation Files\Extracted Features\Set I 13.08.2024 - 15.08.2024\SC_features",
        "rolloff": r"X:\Dissertation Files\Extracted Features\Set I 13.08.2024 - 15.08.2024\Rolloff_features",
        "env"    : r"X:\Dissertation Files\Extracted Features\Set I 13.08.2024 - 15.08.2024\Env_logs",
        "local_tz": "Europe/Warsaw",
    },
    "Set II (13.09.2024)": {
        "rms"    : r"X:\Dissertation Files\Extracted Features\Set II 13.09.2024\RMS_features",
        "zcr"    : r"X:\Dissertation Files\Extracted Features\Set II 13.09.2024\ZCR_features",
        "sb"     : r"X:\Dissertation Files\Extracted Features\Set II 13.09.2024\SB_features",
        "sc"     : r"X:\Dissertation Files\Extracted Features\Set II 13.09.2024\SC_features",
        "rolloff": r"X:\Dissertation Files\Extracted Features\Set II 13.09.2024\Rolloff_features",
        "env"    : r"X:\Dissertation Files\Extracted Features\Set II 13.09.2024\Env_logs",
        "local_tz": "Europe/Warsaw",
    },
    "Set III (21.09.2024-23.09.2024)": {
        "rms"    : r"X:\Dissertation Files\Extracted Features\Set II 21.09.2024 - 23.09.2024\RMS_features",
        "zcr"    : r"X:\Dissertation Files\Extracted Features\Set II 21.09.2024 - 23.09.2024\ZCR_features",
        "sb"     : r"X:\Dissertation Files\Extracted Features\Set II 21.09.2024 - 23.09.2024\SB_features",
        "sc"     : r"X:\Dissertation Files\Extracted Features\Set II 21.09.2024 - 23.09.2024\SC_features",
        "rolloff": r"X:\Dissertation Files\Extracted Features\Set II 21.09.2024 - 23.09.2024\Rolloff_features",
        "env"    : r"X:\Dissertation Files\Extracted Features\Set II 21.09.2024 - 23.09.2024\Env_logs",
        "local_tz": "Europe/Warsaw",
    },
    "Set IV (24.09.2024-26.09.2024)": {
        "rms"    : r"X:\Dissertation Files\Extracted Features\Set II 24.09.2024 - 26.09.2024\RMS_features",
        "zcr"    : r"X:\Dissertation Files\Extracted Features\Set II 24.09.2024 - 26.09.2024\ZCR_features",
        "sb"     : r"X:\Dissertation Files\Extracted Features\Set II 24.09.2024 - 26.09.2024\SB_features",
        "sc"     : r"X:\Dissertation Files\Extracted Features\Set II 24.09.2024 - 26.09.2024\SC_features",
        "rolloff": r"X:\Dissertation Files\Extracted Features\Set II 24.09.2024 - 26.09.2024\Rolloff_features",
        "env"    : r"X:\Dissertation Files\Extracted Features\Set II 24.09.2024 - 26.09.2024\Env_logs",
        "local_tz": "Europe/Warsaw",
    },
    "Set V (30.06.2025-01.07.2025)": {
        "rms"    : r"X:\Dissertation Files\Extracted Features\Set III 30.06.2025 - 01.07.2025\RMS_features",
        "zcr"    : r"X:\Dissertation Files\Extracted Features\Set III 30.06.2025 - 01.07.2025\ZCR_features",
        "sb"     : r"X:\Dissertation Files\Extracted Features\Set III 30.06.2025 - 01.07.2025\SB_features",
        "sc"     : r"X:\Dissertation Files\Extracted Features\Set III 30.06.2025 - 01.07.2025\SC_features",
        "rolloff": r"X:\Dissertation Files\Extracted Features\Set III 30.06.2025 - 01.07.2025\Rolloff_features",
        "env"    : r"X:\Dissertation Files\Extracted Features\Set III 30.06.2025 - 01.07.2025\Env_logs",
        "local_tz": "Europe/Warsaw",
    },
}

# 4. Timestamp parsing & normalization of columns

In [4]:
def _maybe_replace_decimal_comma(s: pd.Series) -> pd.Series:
    if s.dtype == object:
        return s.str.replace(",", ".", regex=False)
    return s

_TZ_PAT = re.compile(r'(Z$)|([+\-]\d{2}:?\d{2}$)', re.IGNORECASE)

def parse_ts(series, tz_policy="utc_naive", local_tz=DEFAULT_LOCAL_TZ) -> pd.Series:
    """
    tz-aware strings -> UTC
    naive strings    -> interpret as local_tz (e.g., Europe/Warsaw), then to UTC
    return tz-naive UTC unless tz_policy == 'aware_utc'
    """
    s = pd.Series(series).copy()
    s = _maybe_replace_decimal_comma(s).astype(str)

    has_tz = s.str.contains(_TZ_PAT)
    out = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")

    if has_tz.any():
        dt_aware = pd.to_datetime(s[has_tz], errors="coerce", utc=True)
        if tz_policy == "aware_utc":
            out.loc[has_tz] = dt_aware
        else:
            out.loc[has_tz] = dt_aware.dt.tz_convert("UTC").dt.tz_localize(None)

    if (~has_tz).any():
        dt_naive = pd.to_datetime(s[~has_tz], errors="coerce")
        dt_local = (dt_naive
                    .dt.tz_localize(local_tz, ambiguous="infer", nonexistent="shift_forward"))
        if tz_policy == "aware_utc":
            out.loc[~has_tz] = dt_local.dt.tz_convert("UTC")
        else:
            out.loc[~has_tz] = dt_local.dt.tz_convert("UTC").dt.tz_localize(None)
    return out

TEMP_NAMES = {"temp","temperature","temp_c","t","°c","degc","air_temp","hive_temp","temperatura (°c)","temperatura"}
HUM_NAMES  = {"humidity","hum","rh","rh_%","rel_humidity","relative_humidity","%rh","wilgotność (%)","wilgotnosc"}

def _standardize_env_columns(df: pd.DataFrame) -> pd.DataFrame:
    cols = {}
    for c in df.columns:
        lc = c.lower().strip()
        if lc in TEMP_NAMES: cols[c] = "temperature"
        elif lc in HUM_NAMES: cols[c] = "humidity"
        else: cols[c] = c
    return df.rename(columns=cols)

FEATURE_SYNONYMS = {
    "rms":     ["rms","root_mean_square","root_mean_sqr","rms_energy","rms_value"],
    "zcr":     ["zcr","zero_crossing_rate","zero_crossing","zero_x"],
    "sb":      ["sb","spectral_bandwidth","bandwidth","spec_bandwidth","spec_bw"],
    "sc":      ["sc","spectral_centroid","centroid","spec_centroid"],
    "rolloff": ["rolloff","roll_off","spectral_rolloff","spec_rolloff","rolloff_85","rolloff_90","rolloff_95"],
}

def _find_timestamp_column(df: pd.DataFrame):
    cands = [c for c in df.columns if c.lower() in ("timestamp","time","datetime","date","date_time")]
    return cands[0] if cands else None

def _choose_feature_column(df: pd.DataFrame, feature_name: str):
    if feature_name in df.columns:
        return df, feature_name
    ignore = {"timestamp","temperature","humidity","index","idx","chunk","frame","file","path","unnamed: 0"}
    numeric_cols = [c for c in df.columns if c.lower() not in ignore and pd.api.types.is_numeric_dtype(df[c])]
    # exact (case-insensitive)
    for c in df.columns:
        if c.lower() == feature_name.lower():
            return df.rename(columns={c: feature_name}), feature_name
    # synonyms
    syns = [s.lower() for s in FEATURE_SYNONYMS.get(feature_name, [])]
    for c in df.columns:
        if c.lower() in syns:
            return df.rename(columns={c: feature_name}), feature_name
    for c in df.columns:
        if any(s in c.lower() for s in syns):
            return df.rename(columns={c: feature_name}), feature_name
    if numeric_cols:
        return df.rename(columns={numeric_cols[0]: feature_name}), feature_name
    return df, None

# 5. Readers (features and environment)

In [5]:
def read_feature_folder(folder: str, feature_name: str, local_tz=DEFAULT_LOCAL_TZ) -> pd.DataFrame:
    files = sorted(
        glob.glob(os.path.join(folder, "*.csv")) +
        glob.glob(os.path.join(folder, "*.CSV")) +
        glob.glob(os.path.join(folder, "*.xlsx")) +
        glob.glob(os.path.join(folder, "*.xls"))
    )
    if not files:
        return pd.DataFrame(columns=["timestamp", feature_name])

    parts = []
    for f in files:
        try:
            df = pd.read_excel(f) if f.lower().endswith((".xlsx",".xls")) else pd.read_csv(f)
        except Exception as e:
            print(f"[WARN] feature read fail: {f}: {e}"); continue
        ts_col = _find_timestamp_column(df)
        if ts_col is None:
            if isinstance(df.index, pd.DatetimeIndex):
                df = df.reset_index().rename(columns={"index":"timestamp"})
            else:
                print(f"[WARN] no timestamp in {f}; skipping."); continue
        df = df.rename(columns={ts_col: "timestamp"})
        df["timestamp"] = parse_ts(df["timestamp"], local_tz=local_tz)
        df = _standardize_env_columns(df)
        df, chosen = _choose_feature_column(df, feature_name)
        if chosen is None:
            print(f"[WARN] no '{feature_name}' column in {f}; skipping."); continue
        d = df[["timestamp", feature_name]].copy()
        d[feature_name] = pd.to_numeric(d[feature_name], errors="coerce")
        d = d.dropna(subset=["timestamp"])
        parts.append(d)

    if not parts:
        return pd.DataFrame(columns=["timestamp", feature_name])
    all_df = pd.concat(parts, ignore_index=True)
    if all_df["timestamp"].duplicated().any():
        all_df = (all_df.groupby("timestamp", as_index=False).agg({feature_name: "mean"}))
    return all_df.sort_values("timestamp").reset_index(drop=True)


def read_env_folder(path_like: str, local_tz=DEFAULT_LOCAL_TZ) -> pd.DataFrame:
    p = Path(path_like)
    files = []
    if p.is_file():
        files = [str(p)]
    else:
        for pat in ("**/*.xlsx","**/*.xls","**/*.csv"):
            files += [str(x) for x in p.glob(pat)]
    if not files:
        return pd.DataFrame(columns=["timestamp","temperature","humidity"])

    parts = []
    for f in sorted(files):
        try:
            df = pd.read_excel(f) if f.lower().endswith((".xlsx",".xls")) else pd.read_csv(f)
        except Exception as e:
            print(f"[WARN] env read fail: {f}: {e}"); continue
        ts_col = _find_timestamp_column(df)
        if ts_col is None:
            if isinstance(df.index, pd.DatetimeIndex):
                df = df.reset_index().rename(columns={"index":"timestamp"})
            else:
                print(f"[WARN] env: no timestamp in {f}; skipping."); continue
        df = df.rename(columns={ts_col: "timestamp"})
        df["timestamp"] = parse_ts(df["timestamp"], local_tz=local_tz)
        df = _standardize_env_columns(df)
        keep = ["timestamp"] + [c for c in ("temperature","humidity") if c in df.columns]
        d = df[keep].copy()
        for c in ("temperature","humidity"):
            if c in d.columns:
                d[c] = pd.to_numeric(d[c], errors="coerce")
        d = d.dropna(subset=["timestamp"])
        parts.append(d)

    if not parts:
        return pd.DataFrame(columns=["timestamp","temperature","humidity"])
    env = (pd.concat(parts, ignore_index=True)
             .dropna(subset=["timestamp"])
             .sort_values("timestamp")
             .reset_index(drop=True))
    vals = [c for c in ("temperature","humidity") if c in env.columns]
    if env["timestamp"].duplicated().any():
        if vals:
            env = (env.groupby("timestamp", as_index=False)[vals].agg("mean"))
        else:
            env = env.drop_duplicates(subset=["timestamp"])
    for c in ("temperature","humidity"):
        if c not in env.columns: env[c] = np.nan
    return env[["timestamp","temperature","humidity"]]

# 6. Alignment utilities and merging

In [6]:
def _resample_feat(df: pd.DataFrame, col: str, rule=RESAMPLE_RULE) -> pd.DataFrame:
    d = df.set_index("timestamp")[[col]].sort_index().resample(rule).mean()
    return d.reset_index()

def _resample_env(df: pd.DataFrame, rule=RESAMPLE_RULE, ffill_limit=ENV_FFILL_LIMIT) -> pd.DataFrame:
    e = df.set_index("timestamp")[["temperature","humidity"]].sort_index().resample(rule).mean()
    e["temperature"] = e["temperature"].ffill(limit=ffill_limit)
    e["humidity"]    = e["humidity"].ffill(limit=ffill_limit)
    return e.reset_index()

def _coverage_vs_base(base_ts: pd.Series, other_ts: pd.Series, tol: pd.Timedelta):
    # ensure datetime and sort
    a = pd.DataFrame({"t": pd.to_datetime(base_ts)}).dropna().sort_values("t")
    b = pd.DataFrame({"t_other": pd.to_datetime(other_ts)}).dropna().sort_values("t_other")

    if a.empty or b.empty:
        return 0.0, None

    # use left_on / right_on so we keep both time columns
    m = pd.merge_asof(
        a, b, left_on="t", right_on="t_other",
        direction="nearest", tolerance=tol
    )

    # coverage = fraction of rows that found a match within tol
    cov = m["t_other"].notna().mean() if len(m) else 0.0

    # offset stats (seconds), absolute value
    if cov == 0:
        return 0.0, None
    dt = (m["t_other"] - m["t"]).dt.total_seconds().abs()
    stats = dt.dropna().describe(percentiles=[.5, .9, .95]).to_dict()
    return cov, stats


def diagnose_session_alignment(name: str, feats: dict, env_df: pd.DataFrame,
                               feat_tol=FEAT_TOL, env_tol=ENV_TOL):
    order = ["rms","zcr","sb","sc","rolloff"]
    base_key = next((k for k in order if k in feats and not feats[k].empty), None)
    print(f"\n[diagnose] {name}")
    if base_key is None:
        print("  no feature streams found"); return
    base = feats[base_key]
    print(f"  base: {base_key} ({len(base)} rows)")
    for k in order:
        if k==base_key or k not in feats: continue
        dfk = feats[k]
        if dfk is None or dfk.empty:
            print(f"  {k}: empty"); continue
        cov, stats = _coverage_vs_base(base["timestamp"], dfk["timestamp"], feat_tol)
        print(f"  {k} vs {base_key}: coverage={cov:.3f} within {feat_tol}, offset stats (s)={stats}")
    if env_df is not None and not env_df.empty:
        cov, stats = _coverage_vs_base(base["timestamp"], env_df["timestamp"], env_tol)
        print(f"  env vs {base_key}: coverage={cov:.3f} within {env_tol}, offset stats (s)={stats}")
    else:
        print("  env: empty")

def merge_features(feat_dfs: dict, env_df: pd.DataFrame | None,
                   align_mode=ALIGN_MODE, feat_tol=FEAT_TOL, env_tol=ENV_TOL) -> pd.DataFrame:
    order = ["rms","zcr","sb","sc","rolloff"]
    base_key = next((k for k in order if k in feat_dfs and not feat_dfs[k].empty), None)
    if base_key is None:
        out = pd.DataFrame({"timestamp":[]}); 
        for c in order + ["temperature","humidity"]: out[c]=[]
        return out

    if align_mode.lower() == "asof":
        base = feat_dfs[base_key][["timestamp", base_key]].sort_values("timestamp").reset_index(drop=True)
        for k in order:
            if k==base_key: continue
            dfk = feat_dfs.get(k)
            if dfk is None or dfk.empty: continue
            right = dfk[["timestamp", k]].sort_values("timestamp")
            base  = pd.merge_asof(base, right, on="timestamp",
                                  direction="nearest", tolerance=feat_tol)
        if env_df is not None and not env_df.empty:
            env_sorted = env_df.sort_values("timestamp")
            base = pd.merge_asof(base, env_sorted, on="timestamp",
                                 direction="nearest", tolerance=env_tol)
        else:
            base["temperature"] = np.nan
            base["humidity"]    = np.nan
        for c in order:
            if c not in base.columns: base[c] = np.nan
        return base.sort_values("timestamp").reset_index(drop=True)

    # resample mode
    starts, ends = [], []
    for k, dfk in feat_dfs.items():
        if dfk is not None and not dfk.empty:
            starts.append(dfk["timestamp"].min()); ends.append(dfk["timestamp"].max())
    if env_df is not None and not env_df.empty:
        starts.append(env_df["timestamp"].min()); ends.append(env_df["timestamp"].max())
    start = pd.to_datetime(min(starts)).floor(RESAMPLE_RULE)
    end   = pd.to_datetime(max(ends)).ceil(RESAMPLE_RULE)
    grid  = pd.DataFrame({"timestamp": pd.date_range(start, end, freq=RESAMPLE_RULE)})

    base = grid.copy()
    for k in order:
        dfk = feat_dfs.get(k)
        if dfk is None or dfk.empty: 
            base[k] = np.nan; continue
        r = _resample_feat(dfk, k, RESAMPLE_RULE)
        base = base.merge(r, on="timestamp", how="left")
    if env_df is not None and not env_df.empty:
        e = _resample_env(env_df, RESAMPLE_RULE, ENV_FFILL_LIMIT)
        base = base.merge(e, on="timestamp", how="left")
    else:
        base["temperature"] = np.nan; base["humidity"] = np.nan
    return base.sort_values("timestamp").reset_index(drop=True)

# 7. Building session and sanity reporting

In [7]:
def build_session_dataframe(paths_dict: dict, session_name="(unnamed)", diagnose=False) -> pd.DataFrame:
    local_tz = paths_dict.get("local_tz", DEFAULT_LOCAL_TZ)

    # read features
    feats = {}
    for key in ["rms","zcr","sb","sc","rolloff"]:
        if key in paths_dict and paths_dict[key]:
            feats[key] = read_feature_folder(paths_dict[key], key, local_tz=local_tz)

    # read env
    env = None
    if "env" in paths_dict and paths_dict["env"]:
        env = read_env_folder(paths_dict["env"], local_tz=local_tz)

    # optional diagnostics
    if diagnose:
        diagnose_session_alignment(session_name, feats, env, FEAT_TOL, ENV_TOL)

    # merge
    df = merge_features(feats, env, ALIGN_MODE, FEAT_TOL, ENV_TOL)

    # numeric coercion safeguard
    for c in ["rms","zcr","sb","sc","rolloff","temperature","humidity"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    return df


def sanity_report(df: pd.DataFrame, title=""):
    print(f"=== {title} ===")
    if df.empty:
        print("EMPTY\n"); return
    print("time span:", df["timestamp"].min(), "→", df["timestamp"].max())
    print("counts:\n", df.count())
    print("missing (%):\n", (df.isna().mean()*100).round(2), "\n")

# 8. Quick file preview

In [8]:
def preview_feature_files():
    for set_name, paths in SESSIONS.items():
        print(f"\n-- {set_name} --")
        for feat in ["rms","zcr","sb","sc","rolloff"]:
            p = paths.get(feat)
            if not p:
                print(f"  {feat}: missing path")
                continue
            files = sorted(
                glob.glob(os.path.join(p, "*.csv")) +
                glob.glob(os.path.join(p, "*.CSV")) +
                glob.glob(os.path.join(p, "*.xlsx")) +
                glob.glob(os.path.join(p, "*.xls"))
            )
            print(f"  {feat}: {len(files)} files | {p}")
            if files:
                try:
                    if files[0].lower().endswith((".xlsx",".xls")):
                        df0 = pd.read_excel(files[0], nrows=3)
                    else:
                        df0 = pd.read_csv(files[0], nrows=3)
                    print("    sample cols:", list(df0.columns))
                except Exception as e:
                    print("    [WARN] cannot open sample:", e)

# Run if you want a quick peek:
preview_feature_files()


-- Set I (13.08.2024-15.08.2024) --
  rms: 30 files | X:\Dissertation Files\Extracted Features\Set I 13.08.2024 - 15.08.2024\RMS_features
    sample cols: ['timestamp', 'rms', 'Temperatura (°C)', 'Wilgotność (%)']
  zcr: 30 files | X:\Dissertation Files\Extracted Features\Set I 13.08.2024 - 15.08.2024\ZCR_features
    sample cols: ['timestamp', 'zcr', 'Temperatura (°C)', 'Wilgotność (%)']
  sb: 30 files | X:\Dissertation Files\Extracted Features\Set I 13.08.2024 - 15.08.2024\SB_features
    sample cols: ['timestamp', 'spectral_bandwidth', 'Temperatura (°C)', 'Wilgotność (%)']
  sc: 30 files | X:\Dissertation Files\Extracted Features\Set I 13.08.2024 - 15.08.2024\SC_features
    sample cols: ['timestamp', 'spectral_centroid', 'Temperatura (°C)', 'Wilgotność (%)', 'Ciśnienie (hPa)', 'RSSI (dBm)', 'Przyspieszenie X (g)', 'Przyspieszenie Y (g)', 'Przyspieszenie Z (g)', 'Napięcie baterii (V)', 'Licznik przesunięć (movements)', 'Numer sekwencji pomiaru', 'Moc nadawania (dBm)']
  rolloff: 30

# 9. Build all sesssions and sanity checks

In [9]:
session_dfs = {}
for name, paths in SESSIONS.items():
    print(f"Building session: {name}")
    df = build_session_dataframe(paths, session_name=name, diagnose=True)
    session_dfs[name] = df
    print(df.head(3))
    print("shape:", df.shape, "\n" + "-"*70)

print("\n=== Sanity checks (raw aligned) ===")
for name, df in session_dfs.items():
    sanity_report(df, name)

# Save aligned CSVs (raw aligned; may still have NaNs in some columns)
for name, df in session_dfs.items():
    safe = name.replace(" ", "_").replace("(", "").replace(")", "").replace(":", "-").replace("/", "-")
    outp = os.path.join(OUTPUT_DIR, f"{safe}.csv")
    df.to_csv(outp, index=False)
    print("saved:", outp, df.shape)

Building session: Set I (13.08.2024-15.08.2024)

[diagnose] Set I (13.08.2024-15.08.2024)
  base: rms (179864 rows)
  zcr vs rms: coverage=1.000 within 0 days 00:00:02, offset stats (s)={'count': 179864.0, 'mean': 0.0, 'std': 0.0, 'min': 0.0, '50%': 0.0, '90%': 0.0, '95%': 0.0, 'max': 0.0}
  sb vs rms: coverage=1.000 within 0 days 00:00:02, offset stats (s)={'count': 179864.0, 'mean': 0.0, 'std': 0.0, 'min': 0.0, '50%': 0.0, '90%': 0.0, '95%': 0.0, 'max': 0.0}
  sc vs rms: coverage=1.000 within 0 days 00:00:02, offset stats (s)={'count': 179864.0, 'mean': 0.0, 'std': 0.0, 'min': 0.0, '50%': 0.0, '90%': 0.0, '95%': 0.0, 'max': 0.0}
  rolloff vs rms: coverage=1.000 within 0 days 00:00:02, offset stats (s)={'count': 179864.0, 'mean': 0.0, 'std': 0.0, 'min': 0.0, '50%': 0.0, '90%': 0.0, '95%': 0.0, 'max': 0.0}
  env vs rms: coverage=0.024 within 0 days 00:02:00, offset stats (s)={'count': 4385.0, 'mean': 59.19053600342075, 'std': 35.10390801887592, 'min': 0.0, '50%': 59.110375, '90%': 107.

saved: X:\Dissertation Files\Extracted Features\_cleaned_for_PCA_vv\Set_I_13.08.2024-15.08.2024.csv (179864, 8)
saved: X:\Dissertation Files\Extracted Features\_cleaned_for_PCA_vv\Set_II_13.09.2024.csv (97348, 8)
saved: X:\Dissertation Files\Extracted Features\_cleaned_for_PCA_vv\Set_III_21.09.2024-23.09.2024.csv (87142, 8)
saved: X:\Dissertation Files\Extracted Features\_cleaned_for_PCA_vv\Set_IV_24.09.2024-26.09.2024.csv (121690, 8)
saved: X:\Dissertation Files\Extracted Features\_cleaned_for_PCA_vv\Set_V_30.06.2025-01.07.2025.csv (86839, 8)


# 10. PCA helpers and plotting

In [10]:
def prepare_matrix_for_pca(df, include_env=False, feature_cols=None, scale=True):
    if feature_cols is None:
        feature_cols = ACOUSTIC_COLS
    cols = [c for c in feature_cols if c in df.columns]
    if include_env:
        for c in ["temperature","humidity"]:
            if c in df.columns:
                cols.append(c)
    valid = df.dropna(subset=cols).copy()
    if valid.empty or len(cols) == 0:
        return None, None, None, None
    X = valid[cols].astype(float).values
    scaler = None
    if scale:
        scaler = StandardScaler()
        Xz = scaler.fit_transform(X)
    else:
        Xz = X
    return Xz, cols, valid, scaler

def fit_pca(Xz, n_components=N_COMPONENTS):
    if n_components is None:
        n_components = min(Xz.shape[1], 6)
    pca = PCA(n_components=n_components, random_state=0)
    scores = pca.fit_transform(Xz)
    return pca, scores

# Matplotlib defaults
plt.rcParams["figure.dpi"] = 140
plt.rcParams["savefig.dpi"] = 140
plt.rcParams["figure.figsize"] = (6, 4)
FIG_DIR = OUTPUT_DIR
Path(FIG_DIR).mkdir(parents=True, exist_ok=True)

def _slug(s: str) -> str:
    return "".join(c if c.isalnum() or c in "-_." else "_" for c in s)

def _savefig(fname: str):
    full = os.path.join(FIG_DIR, _slug(fname))
    plt.savefig(full, bbox_inches="tight"); plt.close()
    print("saved fig:", full)

def plot_scree(pca, title, fname):
    vr = pca.explained_variance_ratio_ * 100
    plt.figure()
    plt.plot(np.arange(1, len(vr)+1), vr, marker='o')
    plt.xlabel('Principal Component'); plt.ylabel('Variance Explained (%)')
    plt.title(title); plt.ylim(0, 100); plt.xticks(np.arange(1, len(vr)+1))
    _savefig(fname)

def plot_loadings(pca, feat_names, pc_index, title_prefix, fname):
    comps = pca.components_[pc_index]
    order = np.argsort(np.abs(comps))[::-1]
    labels = [feat_names[i] for i in order]; vals = comps[order]
    plt.figure(); plt.barh(labels[::-1], vals[::-1])
    plt.title(f"{title_prefix}: PC{pc_index+1}"); plt.xlabel('Loading'); plt.xlim(-1,1)
    _savefig(fname)

def plot_scatter_categorical(scores, labels, title, xlim, ylim, fname):
    x, y = scores[:,0], scores[:,1]
    labels = pd.Series(labels).astype('category'); cats = labels.cat.categories
    plt.figure()
    for cat in cats:
        idx = (labels == cat).values
        plt.scatter(x[idx], y[idx], s=12, label=str(cat))
    plt.axhline(0, color='k', lw=0.5, alpha=0.4); plt.axvline(0, color='k', lw=0.5, alpha=0.4)
    if xlim: plt.xlim(*xlim); 
    if ylim: plt.ylim(*ylim)
    plt.xlabel('PC1'); plt.ylabel('PC2'); plt.title(title); plt.legend(loc='best', frameon=True)
    _savefig(fname)

def plot_scatter_numeric(scores, values, title, xlim, ylim, vmin, vmax, fname):
    x, y = scores[:,0], scores[:,1]
    plt.figure()
    sc = plt.scatter(x, y, c=values, s=10, vmin=vmin, vmax=vmax)
    cb = plt.colorbar(sc); cb.set_label('Value')
    plt.axhline(0, color='k', lw=0.5, alpha=0.4); plt.axvline(0, color='k', lw=0.5, alpha=0.4)
    if xlim: plt.xlim(*xlim); 
    if ylim: plt.ylim(*ylim)
    plt.xlabel('PC1'); plt.ylabel('PC2'); plt.title(title)
    _savefig(fname)

def plot_pc_time_series(valid_df, scores, which_pc, title_prefix, ylim, fname):
    pc_idx = which_pc - 1
    plt.figure(figsize=(10,4))
    plt.plot(valid_df['timestamp'].values, scores[:, pc_idx], lw=0.8)
    plt.title(f"{title_prefix}: PC{which_pc}")
    plt.xlabel('Time'); plt.ylabel(f'PC{which_pc} score')
    if ylim: plt.ylim(*ylim)
    _savefig(fname)

def _minmax(arrs):
    if not arrs: return None
    a = np.concatenate(arrs)
    lo, hi = np.nanmin(a), np.nanmax(a)
    pad = 0.05 * (hi - lo) if hi > lo else 1.0
    return (lo - pad, hi + pad)

# 11. Fitting PCA per session (global axes)

In [11]:
session_results = {}
pc1_all, pc2_all = [], []
all_temp_vals, all_hum_vals = [], []

print("=== PCA fit per session ===")
for name, df in session_dfs.items():
    Xz, feat_names, valid, scaler = prepare_matrix_for_pca(
        df,
        include_env=INCLUDE_ENV_IN_PCA,
        feature_cols=ACOUSTIC_COLS,
        scale=SCALE_BEFORE_PCA
    )
    if Xz is None or valid is None or Xz.shape[0] < 10 or Xz.shape[1] < 2:
        print(f"[skip] {name}: not enough clean rows/features for PCA (rows={0 if Xz is None else Xz.shape[0]})")
        continue

    pca, scores = fit_pca(Xz, N_COMPONENTS)
    session_results[name] = {"pca":pca, "scores":scores, "valid":valid, "feat_names":feat_names}

    pc1_all.append(scores[:,0])
    if pca.n_components_ >= 2: pc2_all.append(scores[:,1])

    if 'temperature' in valid.columns: all_temp_vals.append(valid['temperature'].dropna().values)
    if 'humidity'   in valid.columns: all_hum_vals.append(valid['humidity'].dropna().values)

    if SAVE_PCA_READY:
        safe = name.replace(" ", "_").replace("(", "").replace(")", "").replace(":", "-").replace("/", "-")
        outp = os.path.join(OUTPUT_DIR, f"{safe}__PCA_ready.csv")
        valid.to_csv(outp, index=False)
        print("saved:", outp, valid.shape)

# Global axes / color ranges
if AXIS_MODE == "global":
    PC12_XLIM = _minmax(pc1_all)
    PC12_YLIM = _minmax(pc2_all) if pc2_all else None
else:
    PC12_XLIM, PC12_YLIM = FIXED_PC1, FIXED_PC2

TEMP_RANGE = _minmax(all_temp_vals)
HUM_RANGE  = _minmax(all_hum_vals)

print("Fixed PC1 xlim:", PC12_XLIM, " | Fixed PC2 ylim:", PC12_YLIM)
print("Fixed TEMP range:", TEMP_RANGE, " | Fixed HUM range:", HUM_RANGE)

=== PCA fit per session ===
saved: X:\Dissertation Files\Extracted Features\_cleaned_for_PCA_vv\Set_I_13.08.2024-15.08.2024__PCA_ready.csv (179864, 8)
saved: X:\Dissertation Files\Extracted Features\_cleaned_for_PCA_vv\Set_II_13.09.2024__PCA_ready.csv (97348, 8)
saved: X:\Dissertation Files\Extracted Features\_cleaned_for_PCA_vv\Set_III_21.09.2024-23.09.2024__PCA_ready.csv (70343, 8)
saved: X:\Dissertation Files\Extracted Features\_cleaned_for_PCA_vv\Set_IV_24.09.2024-26.09.2024__PCA_ready.csv (121690, 8)
saved: X:\Dissertation Files\Extracted Features\_cleaned_for_PCA_vv\Set_V_30.06.2025-01.07.2025__PCA_ready.csv (86721, 8)
Fixed PC1 xlim: (-19.361919702528414, 34.46415864114751)  | Fixed PC2 ylim: (-10.431829294443922, 44.75712514627855)
Fixed TEMP range: (9.071, 37.209)  | Fixed HUM range: (32.056999999999995, 74.363)


# 12. Plots per session

In [12]:
for name, res in session_results.items():
    pca, scores, valid, feats = res["pca"], res["scores"], res["valid"], res["feat_names"]

    # Scree + loadings
    plot_scree(pca, title=f"{name} — Scree", fname=f"{name}__scree.png")
    plot_loadings(pca, feats, pc_index=0, title_prefix=f"{name}", fname=f"{name}__loadings_PC1.png")
    if pca.n_components_ > 1:
        plot_loadings(pca, feats, pc_index=1, title_prefix=f"{name}", fname=f"{name}__loadings_PC2.png")

    # PC1 vs PC2 by session (categorical)
    if pca.n_components_ >= 2:
        plot_scatter_categorical(scores, [name]*len(valid),
                                 title=f"{name} — PC1 vs PC2 by SESSION",
                                 xlim=PC12_XLIM, ylim=PC12_YLIM,
                                 fname=f"{name}__PC1vPC2_bySESSION.png")

    # PC1 vs PC2 by environment (if present)
    if pca.n_components_ >= 2 and 'temperature' in valid.columns and valid['temperature'].notna().any():
        plot_scatter_numeric(scores, valid['temperature'].values,
                             title=f"{name} — PC1 vs PC2 by TEMP",
                             xlim=PC12_XLIM, ylim=PC12_YLIM,
                             vmin=(None if TEMP_RANGE is None else TEMP_RANGE[0]),
                             vmax=(None if TEMP_RANGE is None else TEMP_RANGE[1]),
                             fname=f"{name}__PC1vPC2_byTEMP.png")

    if pca.n_components_ >= 2 and 'humidity' in valid.columns and valid['humidity'].notna().any():
        plot_scatter_numeric(scores, valid['humidity'].values,
                             title=f"{name} — PC1 vs PC2 by HUM",
                             xlim=PC12_XLIM, ylim=PC12_YLIM,
                             vmin=(None if HUM_RANGE is None else HUM_RANGE[0]),
                             vmax=(None if HUM_RANGE is None else HUM_RANGE[1]),
                             fname=f"{name}__PC1vPC2_byHUM.png")

    # Time series (global y-limits for consistency)
    TS_PC1_YLIM = _minmax(pc1_all)
    TS_PC2_YLIM = _minmax(pc2_all) if pc2_all else None
    plot_pc_time_series(valid, scores, which_pc=1, title_prefix=f"{name}", ylim=TS_PC1_YLIM, fname=f"{name}__PC1_timeseries.png")
    if pca.n_components_ > 1:
        plot_pc_time_series(valid, scores, which_pc=2, title_prefix=f"{name}", ylim=TS_PC2_YLIM, fname=f"{name}__PC2_timeseries.png")

saved fig: X:\Dissertation Files\Extracted Features\_cleaned_for_PCA_vv\Set_I__13.08.2024-15.08.2024___scree.png
saved fig: X:\Dissertation Files\Extracted Features\_cleaned_for_PCA_vv\Set_I__13.08.2024-15.08.2024___loadings_PC1.png
saved fig: X:\Dissertation Files\Extracted Features\_cleaned_for_PCA_vv\Set_I__13.08.2024-15.08.2024___loadings_PC2.png
saved fig: X:\Dissertation Files\Extracted Features\_cleaned_for_PCA_vv\Set_I__13.08.2024-15.08.2024___PC1vPC2_bySESSION.png
saved fig: X:\Dissertation Files\Extracted Features\_cleaned_for_PCA_vv\Set_I__13.08.2024-15.08.2024___PC1vPC2_byTEMP.png
saved fig: X:\Dissertation Files\Extracted Features\_cleaned_for_PCA_vv\Set_I__13.08.2024-15.08.2024___PC1vPC2_byHUM.png
saved fig: X:\Dissertation Files\Extracted Features\_cleaned_for_PCA_vv\Set_I__13.08.2024-15.08.2024___PC1_timeseries.png
saved fig: X:\Dissertation Files\Extracted Features\_cleaned_for_PCA_vv\Set_I__13.08.2024-15.08.2024___PC2_timeseries.png
saved fig: X:\Dissertation Files\Ex

# 13. Combined PCA 

In [13]:
combined = []
for name, df in session_dfs.items():
    if not df.empty:
        d = df.copy(); d["session"] = name; combined.append(d)

if not combined:
    print("No sessions to combine.")
else:
    all_df = pd.concat(combined, ignore_index=True).sort_values('timestamp').reset_index(drop=True)
    Xz_all, feat_names_all, valid_all, scaler_all = prepare_matrix_for_pca(
        all_df, include_env=INCLUDE_ENV_IN_PCA, feature_cols=ACOUSTIC_COLS, scale=SCALE_BEFORE_PCA
    )
    if Xz_all is None or Xz_all.shape[0] < 10 or Xz_all.shape[1] < 2:
        print("Combined: not enough clean rows/features for PCA.")
    else:
        pca_all, scores_all = fit_pca(Xz_all, N_COMPONENTS)
        COMB_XLIM = _minmax([scores_all[:,0]])
        COMB_YLIM = _minmax([scores_all[:,1]]) if pca_all.n_components_ >= 2 else None

        plot_scree(pca_all, title="Combined — Scree", fname="Combined__scree.png")
        plot_loadings(pca_all, feat_names_all, pc_index=0, title_prefix="Combined", fname="Combined__loadings_PC1.png")
        if pca_all.n_components_ > 1:
            plot_loadings(pca_all, feat_names_all, pc_index=1, title_prefix="Combined", fname="Combined__loadings_PC2.png")

        if pca_all.n_components_ >= 2:
            plot_scatter_categorical(scores_all, valid_all['session'],
                                     title="Combined — PC1 vs PC2 by SESSION",
                                     xlim=COMB_XLIM, ylim=COMB_YLIM,
                                     fname="Combined__PC1vPC2_bySESSION.png")
            if 'temperature' in valid_all.columns and valid_all['temperature'].notna().any():
                plot_scatter_numeric(scores_all, valid_all['temperature'],
                                     title="Combined — PC1 vs PC2 by TEMP",
                                     xlim=COMB_XLIM, ylim=COMB_YLIM,
                                     vmin=(None if TEMP_RANGE is None else TEMP_RANGE[0]),
                                     vmax=(None if TEMP_RANGE is None else TEMP_RANGE[1]),
                                     fname="Combined__PC1vPC2_byTEMP.png")
            if 'humidity' in valid_all.columns and valid_all['humidity'].notna().any():
                plot_scatter_numeric(scores_all, valid_all['humidity'],
                                     title="Combined — PC1 vs PC2 by HUM",
                                     xlim=COMB_XLIM, ylim=COMB_YLIM,
                                     vmin=(None if HUM_RANGE is None else HUM_RANGE[0]),
                                     vmax=(None if HUM_RANGE is None else HUM_RANGE[1]),
                                     fname="Combined__PC1vPC2_byHUM.png")

        TS1 = _minmax([scores_all[:,0]])
        plot_pc_time_series(valid_all, scores_all, which_pc=1, title_prefix="Combined", ylim=TS1, fname="Combined__PC1_timeseries.png")
        if pca_all.n_components_ > 1:
            TS2 = _minmax([scores_all[:,1]])
            plot_pc_time_series(valid_all, scores_all, which_pc=2, title_prefix="Combined", ylim=TS2, fname="Combined__PC2_timeseries.png")

saved fig: X:\Dissertation Files\Extracted Features\_cleaned_for_PCA_vv\Combined__scree.png
saved fig: X:\Dissertation Files\Extracted Features\_cleaned_for_PCA_vv\Combined__loadings_PC1.png
saved fig: X:\Dissertation Files\Extracted Features\_cleaned_for_PCA_vv\Combined__loadings_PC2.png
saved fig: X:\Dissertation Files\Extracted Features\_cleaned_for_PCA_vv\Combined__PC1vPC2_bySESSION.png
saved fig: X:\Dissertation Files\Extracted Features\_cleaned_for_PCA_vv\Combined__PC1vPC2_byTEMP.png
saved fig: X:\Dissertation Files\Extracted Features\_cleaned_for_PCA_vv\Combined__PC1vPC2_byHUM.png
saved fig: X:\Dissertation Files\Extracted Features\_cleaned_for_PCA_vv\Combined__PC1_timeseries.png
saved fig: X:\Dissertation Files\Extracted Features\_cleaned_for_PCA_vv\Combined__PC2_timeseries.png


# 14. Diagnose one session explicity

In [14]:
# session name exactly as in SESSIONS keys:
SESSION_NAME = "Set I (13.08.2024-15.08.2024)"

# Build its raw components to inspect coverage more deeply:
paths = SESSIONS[SESSION_NAME]
local_tz = paths.get("local_tz", DEFAULT_LOCAL_TZ)

feats_dbg = {}
for key in ["rms","zcr","sb","sc","rolloff"]:
    if key in paths and paths[key]:
        feats_dbg[key] = read_feature_folder(paths[key], key, local_tz=local_tz)
env_dbg = read_env_folder(paths["env"], local_tz=local_tz) if "env" in paths and paths["env"] else None

diagnose_session_alignment(SESSION_NAME, feats_dbg, env_dbg, FEAT_TOL, ENV_TOL)



[diagnose] Set I (13.08.2024-15.08.2024)
  base: rms (179864 rows)
  zcr vs rms: coverage=1.000 within 0 days 00:00:02, offset stats (s)={'count': 179864.0, 'mean': 0.0, 'std': 0.0, 'min': 0.0, '50%': 0.0, '90%': 0.0, '95%': 0.0, 'max': 0.0}
  sb vs rms: coverage=1.000 within 0 days 00:00:02, offset stats (s)={'count': 179864.0, 'mean': 0.0, 'std': 0.0, 'min': 0.0, '50%': 0.0, '90%': 0.0, '95%': 0.0, 'max': 0.0}
  sc vs rms: coverage=1.000 within 0 days 00:00:02, offset stats (s)={'count': 179864.0, 'mean': 0.0, 'std': 0.0, 'min': 0.0, '50%': 0.0, '90%': 0.0, '95%': 0.0, 'max': 0.0}
  rolloff vs rms: coverage=1.000 within 0 days 00:00:02, offset stats (s)={'count': 179864.0, 'mean': 0.0, 'std': 0.0, 'min': 0.0, '50%': 0.0, '90%': 0.0, '95%': 0.0, 'max': 0.0}
  env vs rms: coverage=0.024 within 0 days 00:02:00, offset stats (s)={'count': 4385.0, 'mean': 59.19053600342075, 'std': 35.10390801887592, 'min': 0.0, '50%': 59.110375, '90%': 107.9110378, '95%': 113.9928624, 'max': 120.0}
